In [1]:
import numpy as np

In [2]:
import numpy as np


def ndisc_to_resistance(Ndisc):
    # Physical constants
    P_Q = 1.6022e-19      # Elementary charge [C]
    M_PI = np.pi

    # JART model parameters (from JART_VCM_v1b_var.h)
    un = 4e-6             # electron mobility [m^2/Vs]
    Nplug = 20            # oxygen vacancy concentration in plug [10^26/m^3]
    zvo = 2               # oxygen vacancy charge number
    lcell = 3             # total length of disc+plug region [nm]
    lvar = 0.4            # length of the disc region (lnew) [nm]
    rvar = 45e-9          # radius of the filament (rnew) [m]
    RTiOx = 650           # series resistance of TiOx layer [Ohm]
    R0 = 719.2437         # resistance at T0 [Ohm]

    # Calculate filament cross-sectional area
    A = M_PI * rvar**2

    # Convert concentrations from [10^26/m^3] to [1/m^3]
    Nreal_SI = Ndisc * 1e26
    Nplug_SI = Nplug * 1e26

    # Calculate disc resistance (variable, depends on Ndisc)
    # From UpdateResistance: Rdisc = lvar * 1e-9 / (Nreal * 1e26 * zvo * P_Q * un * A)
    Rdisc = (lvar * 1e-9) / (Nreal_SI * zvo * P_Q * un * A)

    # Calculate plug resistance (fixed)
    # From UpdateResistance: Rplug = (lcell - lvar) * 1e-9 / (Nplug * 1e26 * zvo * P_Q * un * A)
    Rplug = ((lcell - lvar) * 1e-9) / (Nplug_SI * zvo * P_Q * un * A)

    # Calculate series resistance (fixed, assuming low current so alphaline term ≈ 0)
    # From UpdateResistance: Rseries = RTiOx + R0 * (1 + R0 * alphaline * I^2 * Rthline)
    Rseries = RTiOx + R0

    # Total resistance
    R_total = Rdisc + Rplug + Rseries

    return R_total


def resistance_to_ndisc(R_target, Ndiscmin=0.0001, Ndiscmax=30.0, method='analytical'):
    # Physical constants and parameters
    P_Q = 1.6022e-19
    M_PI = np.pi
    un = 4e-6
    Nplug = 20
    zvo = 2
    lcell = 3
    lvar = 0.4
    rvar = 45e-9
    RTiOx = 650
    R0 = 719.2437

    A = M_PI * rvar**2
    Nplug_SI = Nplug * 1e26
    Rplug = ((lcell - lvar) * 1e-9) / (Nplug_SI * zvo * P_Q * un * A)
    Rseries = RTiOx + R0

    # Check if target is in valid range
    R_min = ndisc_to_resistance(Ndiscmax)
    R_max = ndisc_to_resistance(Ndiscmin)

    if R_target < R_min or R_target > R_max:
        print(f"Warning: Target resistance {R_target:.2f} Ω is out of range [{R_min:.2f}, {R_max:.2f}] Ω")
        return None

    if method == 'analytical':
        # Analytical solution: R_total = Rdisc + Rplug + Rseries
        # Rdisc = (lvar * 1e-9) / (Ndisc * 1e26 * zvo * P_Q * un * A)
        # Solving for Ndisc:
        Rdisc_target = R_target - Rplug - Rseries

        if Rdisc_target <= 0:
            return Ndiscmax  # Maximum conductivity case

        Ndisc = (lvar * 1e-9) / (Rdisc_target * zvo * P_Q * un * A * 1e26)

        # Clip to valid range
        Ndisc = np.clip(Ndisc, Ndiscmin, Ndiscmax)
        return Ndisc

    else:  # numerical method
        from scipy.optimize import brentq

        def objective(ndisc):
            return ndisc_to_resistance(ndisc) - R_target

        try:
            ndisc = brentq(objective, Ndiscmin, Ndiscmax, xtol=1e-9)
            return ndisc
        except ValueError:
            return None
resistance_to_ndisc(1666)

3.5720292968008014

In [3]:
currents = np.arange(20,80,20)
print(currents)
# currents = [29.4,41.12,62.6]
currents = [20.3,40,62.6]
for I in currents:
    R = 0.1/(I*1e-6)
    Ninit = resistance_to_ndisc(R)
    print(R,Ninit)

[20 40 60]
4926.108374384237 0.14438659405122026
2500.0000000000005 0.5050238227882825
1597.444089456869 7.132760341342812


In [4]:
import random
random.randint(0,3)

0